# Inventory Stock Risk Prediction
## End-to-end classification project

**Objective:** classify inventory records as stockout or overstock risk using operational and demand-related features.

Workflow: data understanding → data quality → EDA → feature engineering → preprocessing → model comparison → tuning → evaluation → model export.

Dataset expected: the supplied Excel workbook (5,000 rows, 11 columns). Place it in this folder or in `data/`.

## 1. Import libraries and load data

In [ ]:
import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
warnings.filterwarnings('ignore')
RANDOM_STATE = 42

candidates = ['stock_risk_prediction_dataset (1).xlsx', 'stock_risk_prediction_dataset.xlsx', os.path.join('data', 'stock_risk_prediction_dataset.xlsx')]
data_path = next((p for p in candidates if os.path.exists(p)), None)
if data_path is None:
    matches = [os.path.join(root, f) for root, _, files in os.walk('.') for f in files if f.lower().endswith(('.xlsx', '.xls')) and 'stock_risk' in f.lower()]
    if not matches:
        raise FileNotFoundError('Place the stock risk Excel workbook in this folder or data/.')
    data_path = matches[0]
df = pd.read_excel(data_path)
print('Dataset shape:', df.shape)
display(df.head())

## 2. Understand and validate the dataset

In [ ]:
target = 'stock_status'
print('Data types:')
display(df.dtypes.to_frame('dtype'))
print('Missing values:')
display(df.isna().sum().to_frame('missing_count'))
print('Duplicate rows:', df.duplicated().sum())
print('Target distribution:')
display(df[target].value_counts())
display((df[target].value_counts(normalize=True) * 100).round(2))
df_clean = df.drop_duplicates().dropna(subset=[target]).copy()
sns.countplot(data=df_clean, x=target)
plt.title('Inventory stock status distribution')
plt.tight_layout()
plt.show()

## 3. Exploratory data analysis

In [ ]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
display(df_clean[numeric_cols].describe().T)
df_clean[numeric_cols].hist(figsize=(14, 9), bins=25)
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 7))
sns.heatmap(df_clean[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numeric feature correlation')
plt.tight_layout()
plt.show()

## 4. Feature engineering and train/test split

The five engineered variables describe inventory coverage, demand during supplier lead time, stock relative to reorder point, absolute forecast error, and average daily sales.

In [ ]:
def add_features(data):
    out = data.copy()
    out['days_of_inventory'] = out['current_stock'] / out['avg_daily_demand'].replace(0, np.nan)
    out['demand_during_lead_time'] = out['avg_daily_demand'] * out['lead_time_days']
    out['stock_vs_reorder_ratio'] = out['current_stock'] / out['reorder_point'].replace(0, np.nan)
    out['forecast_abs_error'] = out['forecast_error'].abs()
    out['sales_per_day_30d'] = out['sales_last_30_days'] / 30
    return out

df_model = add_features(df_clean)
X = df_model.drop(columns=[target])
y = df_model[target].map({'stockout': 1, 'overstock': 0})
if y.isna().any():
    raise ValueError('Unexpected target labels. Expected stockout and overstock.')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()
preprocessor = ColumnTransformer([('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Target counts:', y.value_counts().to_dict())

## 5. Compare classification models

The stockout class is the minority class (about 9.14% in the supplied data). Review precision, recall and F1-score, not accuracy alone.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE, max_depth=8),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}
results, fitted_pipelines = [], {}
for name, estimator in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, pred), 'Precision': precision_score(y_test, pred, zero_division=0), 'Recall': recall_score(y_test, pred, zero_division=0), 'F1': f1_score(y_test, pred, zero_division=0), 'ROC_AUC': roc_auc_score(y_test, prob)})
    fitted_pipelines[name] = pipe
results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
display(results_df.style.format({c: '{:.3f}' for c in results_df.columns if c != 'Model'}))

## 6. Tune the Random Forest and evaluate the final model

In [ ]:
rf_pipeline = Pipeline([('preprocessor', preprocessor), ('model', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
param_grid = {'model__n_estimators': [200, 300], 'model__max_depth': [None, 8, 12], 'model__min_samples_split': [2, 5], 'model__min_samples_leaf': [1, 2]}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(rf_pipeline, param_distributions=param_grid, scoring='f1', cv=cv, n_iter=10, n_jobs=-1, random_state=RANDOM_STATE, verbose=1)
search.fit(X_train, y_train)
final_model = search.best_estimator_
final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, list(final_model.classes_).index(1)]
print('Best parameters:', search.best_params_)
print('Best CV F1:', round(search.best_score_, 4))
print('Hold-out ROC-AUC:', round(roc_auc_score(y_test, final_prob), 4))
print(classification_report(y_test, final_pred, target_names=['overstock', 'stockout'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, final_pred, display_labels=['overstock', 'stockout'])
plt.title('Tuned Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Feature importance and save the model

In [ ]:
feature_names = final_model.named_steps['preprocessor'].get_feature_names_out()
importances = final_model.named_steps['model'].feature_importances_
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False)
display(importance_df.head(15))
importance_df.head(15).sort_values('Importance').plot(kind='barh', x='Feature', y='Importance', legend=False, figsize=(10, 6))
plt.title('Top feature importances — Random Forest')
plt.tight_layout()
plt.show()
joblib.dump(final_model, 'stock_risk_model.pkl')
print('Saved model to stock_risk_model.pkl')

## 8. Business interpretation and limitations

- **Stockout risk:** review replenishment timing, safety stock and supplier lead time.
- **Overstock risk:** review purchasing quantities, demand assumptions and inventory movement.
- Stockout recall is important because missed stockouts can result in lost sales.
- The supplied dataset is limited to 5,000 observations and may not represent all inventory settings.
- The unusually high metrics recorded in the original notebook should be investigated for target leakage and tested on genuinely unseen, time-separated operational data before production use.

Predictions are decision-support signals and should be reviewed alongside operational knowledge.